(Dashboard Financeiro)

In [23]:
# ============================================================
# Item 6 (v6) — Dashboard Interativo Completo — Célula Única e Definitiva
# API completa (todos os endpoints) + Assistente + Gráficos ao vivo +
# Testes + Solicitação de chave embutida + Deploy, tudo em um só lugar
# ============================================================
"""
Versão consolidada: nenhuma etapa separada. Escreve o dashboard.html
(assistente de perguntas + Chart.js + pulso vital animado) e o main.py
completo (categoria, perfil, recomendacoes, alertas, historico, lote
CSV, dashboard), testa localmente, solicita a chave SSH no meio da
execucao se necessario, e faz o deploy completo para producao.

O caminho do dashboard.html e relativo a localizacao do main.py
(os.path.dirname(__file__)), entao funciona tanto localmente no Colab
(/content/) quanto em producao (/home/ubuntu/) sem qualquer ajuste manual.
"""

import subprocess
import sys
import os
import time
import base64


def instalar_dependencias() -> None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "fastapi", "uvicorn", "pytest", "scikit-learn", "pandas", "numpy",
         "joblib", "requests", "python-multipart", "paramiko", "--quiet"],
        check=True
    )
    print("✅ Dependências instaladas (incluindo paramiko)")


instalar_dependencias()
import paramiko
import requests


DASHBOARD_HTML_B64 = "..."


CONTEUDO_MAIN = '''
import os
import json
import sqlite3
import io
import joblib
import numpy as np
import pandas as pd
import requests
from datetime import datetime, timezone
from contextlib import asynccontextmanager
from typing import Literal, Optional
from fastapi import FastAPI, HTTPException, Query, UploadFile, File
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field


class Transacao(BaseModel):
    descricao: str
    valor: float = Field(gt=0)


class AnaliseFinanceiraRequest(BaseModel):
    renda_mensal: float = Field(gt=0)
    nivel_endividamento: float = Field(ge=0, le=100)
    frequencia_poupanca: Literal["Baixa", "Media", "Alta"]
    transacoes: list[Transacao] = Field(min_length=1)


class ClassificarTransacoesRequest(BaseModel):
    transacoes: list[Transacao] = Field(min_length=1)


NOMES_EDITAL = {
    "Alimentacao": "alimentacao", "Moradia": "moradia", "Transporte": "transporte",
    "Saude": "saude", "Educacao": "educacao", "Lazer": "entretenimento", "Servicos": "servicos",
}

FEATURES_MODELO_PERFIL = [
    "renda_mensal", "nivel_endividamento", "frequencia_poupanca_cod", "comprometimento_gastos",
    "Alimentacao", "Moradia", "Transporte", "Saude", "Educacao", "Lazer", "Servicos",
]

LIMIAR_CATEGORIA_ELEVADA = 0.30
LIMIAR_COMPROMETIMENTO_TOTAL = 0.90
COLUNAS_CSV_OBRIGATORIAS = ["usuario_id", "renda_mensal", "nivel_endividamento", "frequencia_poupanca", "descricao", "valor"]

URLS_OCI = {
    "vetorizador_tfidf.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/hLaYP5zlH3YNL_X1uTG7L6P2V4P_DdFRlTCuydgLM8cFpKGlHPrtj0hqvkLFtNQo/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/vetorizador_tfidf.pkl",
    "modelo_categoria_producao.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/TN6b3FeWkczu1zaQvNyiKqhjivp01Orlz0O28TDmR1wM_V6gZUFKtogP8ixqkfH4/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/modelo_categoria_producao.pkl",
    "codificador_categorias.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/_9KWGLcV8EKJ-_EiqlX2kZd78GhIO3lVkqBwNYpWfKrNSU-qWuFgGFDAdw6svF42/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/codificador_categorias.pkl",
    "modelo_perfil_producao.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/yKaMqbJdI3GNQdB76fIY9pbiMaEfh8l4WE4sVehs5AeY2vTxyLILTv611OVXXc57/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/modelo_perfil_producao.pkl",
    "codificador_perfil.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/DDF588_TNDMhg5v7KBnti9DCqKFd4SI_l3WgyIXkmCJkXUlqCqzJWUnWXSQIHru4/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/codificador_perfil.pkl",
}

PASTA_MODELOS = "/home/ubuntu/modelos_api"
CAMINHO_BANCO_DADOS = "/home/ubuntu/historico.db"
CAMINHO_DASHBOARD_HTML = os.path.join(os.path.dirname(os.path.abspath(__file__)), "dashboard.html")


def baixar_e_carregar_artefatos():
    os.makedirs(PASTA_MODELOS, exist_ok=True)
    for nome_arquivo, url in URLS_OCI.items():
        resposta = requests.get(url)
        resposta.raise_for_status()
        with open(os.path.join(PASTA_MODELOS, nome_arquivo), "wb") as f:
            f.write(resposta.content)
    return {
        "vetorizador_tfidf": joblib.load(os.path.join(PASTA_MODELOS, "vetorizador_tfidf.pkl")),
        "modelo_categoria": joblib.load(os.path.join(PASTA_MODELOS, "modelo_categoria_producao.pkl")),
        "codificador_categorias": joblib.load(os.path.join(PASTA_MODELOS, "codificador_categorias.pkl")),
        "modelo_perfil": joblib.load(os.path.join(PASTA_MODELOS, "modelo_perfil_producao.pkl")),
        "codificador_perfil": joblib.load(os.path.join(PASTA_MODELOS, "codificador_perfil.pkl")),
    }


def classificar_categorias_transacoes(transacoes, artefatos):
    descricoes = [t["descricao"] for t in transacoes]
    vetores = artefatos["vetorizador_tfidf"].transform(descricoes)
    categorias_cod = artefatos["modelo_categoria"].predict(vetores)
    categorias_internas = artefatos["codificador_categorias"].inverse_transform(categorias_cod)
    resultado = []
    for t, cat_interna in zip(transacoes, categorias_internas):
        resultado.append({**t, "categoria": NOMES_EDITAL.get(cat_interna, cat_interna.lower()), "_categoria_interna": cat_interna})
    return resultado


def calcular_resumo_gastos(transacoes_classificadas):
    categorias_possiveis = list(NOMES_EDITAL.values())
    resumo = {cat: 0.0 for cat in categorias_possiveis}
    for t in transacoes_classificadas:
        resumo[t["categoria"]] += t["valor"]
    return {cat: round(valor, 2) for cat, valor in resumo.items()}


def calcular_resumo_gastos_interno(transacoes_classificadas):
    categorias_internas = list(NOMES_EDITAL.keys())
    resumo = {cat: 0.0 for cat in categorias_internas}
    for t in transacoes_classificadas:
        resumo[t["_categoria_interna"]] += t["valor"]
    return resumo


def prever_perfil_financeiro(renda_mensal, nivel_endividamento, frequencia_poupanca, resumo_gastos_interno, artefatos):
    mapa_poupanca = {"Baixa": 0, "Media": 1, "Alta": 2}
    comprometimento_gastos = (sum(resumo_gastos_interno.values()) / renda_mensal) * 100
    features = pd.DataFrame([{
        "renda_mensal": renda_mensal, "nivel_endividamento": nivel_endividamento,
        "frequencia_poupanca_cod": mapa_poupanca[frequencia_poupanca],
        "comprometimento_gastos": comprometimento_gastos, **resumo_gastos_interno,
    }])
    probabilidades = artefatos["modelo_perfil"].predict_proba(features)[0]
    indice = np.argmax(probabilidades)
    perfil = artefatos["codificador_perfil"].classes_[indice]
    return perfil, round(float(probabilidades[indice]), 2)


def gerar_recomendacoes(perfil, resumo_gastos, frequencia_poupanca):
    if not resumo_gastos:
        return ["Nenhuma transacao informada para gerar recomendacoes especificas."]
    categoria_maior_gasto = max(resumo_gastos, key=resumo_gastos.get)
    if perfil == "Em risco":
        return [
            f"Reduzir gastos com {categoria_maior_gasto}, categoria de maior peso no orcamento",
            "Buscar renegociacao de dividas para reduzir o nivel de endividamento",
        ]
    if perfil == "Em observacao":
        recs = [f"Monitorar gastos recorrentes de {categoria_maior_gasto}"]
        if frequencia_poupanca == "Baixa":
            recs.append("Aumentar a frequencia de poupanca mensal")
        return recs
    return [
        "Manter o padrao atual de organizacao financeira",
        "Considerar investir o excedente mensal para objetivos de longo prazo",
    ]


def gerar_alertas_gastos_elevados(renda_mensal, resumo_gastos):
    alertas = []
    for categoria, valor in resumo_gastos.items():
        percentual = valor / renda_mensal
        if percentual >= LIMIAR_CATEGORIA_ELEVADA:
            alertas.append({
                "tipo": "categoria_elevada", "categoria": categoria, "valor": valor,
                "percentual_da_renda": round(percentual * 100, 1),
                "mensagem": f"Gasto com {categoria} representa {round(percentual * 100, 1)}% da renda mensal (limiar: {int(LIMIAR_CATEGORIA_ELEVADA * 100)}%)",
            })
    comprometimento_total = sum(resumo_gastos.values()) / renda_mensal
    if comprometimento_total >= LIMIAR_COMPROMETIMENTO_TOTAL:
        alertas.append({
            "tipo": "comprometimento_total_elevado", "categoria": None,
            "valor": round(sum(resumo_gastos.values()), 2), "percentual_da_renda": round(comprometimento_total * 100, 1),
            "mensagem": f"Gasto total representa {round(comprometimento_total * 100, 1)}% da renda mensal (limiar: {int(LIMIAR_COMPROMETIMENTO_TOTAL * 100)}%)",
        })
    return alertas


def analisar_financas(dados_entrada, artefatos):
    transacoes_classificadas = classificar_categorias_transacoes(dados_entrada["transacoes"], artefatos)
    resumo_gastos_edital = calcular_resumo_gastos(transacoes_classificadas)
    resumo_gastos_interno = calcular_resumo_gastos_interno(transacoes_classificadas)
    perfil, probabilidade = prever_perfil_financeiro(
        dados_entrada["renda_mensal"], dados_entrada["nivel_endividamento"],
        dados_entrada["frequencia_poupanca"], resumo_gastos_interno, artefatos
    )
    recomendacoes = gerar_recomendacoes(perfil, resumo_gastos_edital, dados_entrada["frequencia_poupanca"])
    resumo_gastos_filtrado = {k: v for k, v in resumo_gastos_edital.items() if v > 0}
    alertas = gerar_alertas_gastos_elevados(dados_entrada["renda_mensal"], resumo_gastos_filtrado)
    return {
        "perfil_financeiro": perfil, "probabilidade": probabilidade,
        "resumo_gastos": resumo_gastos_filtrado, "recomendacoes": recomendacoes, "alertas": alertas,
    }


def obter_importancia_variaveis_perfil(artefatos):
    modelo = artefatos["modelo_perfil"]
    importancias = modelo.feature_importances_
    pares = list(zip(FEATURES_MODELO_PERFIL, importancias))
    pares_ordenados = sorted(pares, key=lambda x: x[1], reverse=True)
    return [{"variavel": nome, "importancia_percentual": round(float(valor) * 100, 2)} for nome, valor in pares_ordenados]


def obter_palavras_influentes_categoria(artefatos, top_n=8):
    modelo = artefatos["modelo_categoria"]
    vocabulario = artefatos["vetorizador_tfidf"].get_feature_names_out()
    classes = artefatos["codificador_categorias"].classes_
    resultado = {}
    for indice_classe, nome_classe_interno in enumerate(classes):
        coeficientes = modelo.coef_[indice_classe]
        indices_top = np.argsort(coeficientes)[::-1][:top_n]
        nome_edital = NOMES_EDITAL.get(nome_classe_interno, nome_classe_interno.lower())
        resultado[nome_edital] = [vocabulario[i] for i in indices_top]
    return resultado


def inicializar_banco_de_dados(caminho_db):
    aspas = chr(39)  # caractere de aspas simples, isolado para evitar aninhamento de aspas
    conexao = sqlite3.connect(caminho_db)
    conexao.execute(
        "CREATE TABLE IF NOT EXISTS analises ("
        "id INTEGER PRIMARY KEY AUTOINCREMENT, data_hora TEXT NOT NULL, "
        f"usuario_id TEXT NOT NULL DEFAULT {aspas}{aspas}, renda_mensal REAL NOT NULL, "
        "nivel_endividamento REAL NOT NULL, frequencia_poupanca TEXT NOT NULL, "
        "transacoes_json TEXT NOT NULL, perfil_financeiro TEXT NOT NULL, "
        "probabilidade REAL NOT NULL, resumo_gastos_json TEXT NOT NULL, "
        f"recomendacoes_json TEXT NOT NULL, alertas_json TEXT NOT NULL DEFAULT {aspas}[]{aspas})"
    )
    for coluna, definicao in [
        ("alertas_json", f"TEXT NOT NULL DEFAULT {aspas}[]{aspas}"),
        ("usuario_id", f"TEXT NOT NULL DEFAULT {aspas}{aspas}"),
    ]:
        try:
            conexao.execute(f"ALTER TABLE analises ADD COLUMN {coluna} {definicao}")
        except sqlite3.OperationalError:
            pass
    conexao.commit()
    conexao.close()


def salvar_analise_no_historico(caminho_db, dados_entrada, resultado, usuario_id=""):
    conexao = sqlite3.connect(caminho_db)
    cursor = conexao.execute(
        "INSERT INTO analises (data_hora, usuario_id, renda_mensal, nivel_endividamento, "
        "frequencia_poupanca, transacoes_json, perfil_financeiro, probabilidade, "
        "resumo_gastos_json, recomendacoes_json, alertas_json) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        (
            datetime.now(timezone.utc).isoformat(), usuario_id,
            dados_entrada["renda_mensal"], dados_entrada["nivel_endividamento"], dados_entrada["frequencia_poupanca"],
            json.dumps(dados_entrada["transacoes"], ensure_ascii=False),
            resultado["perfil_financeiro"], resultado["probabilidade"],
            json.dumps(resultado["resumo_gastos"], ensure_ascii=False),
            json.dumps(resultado["recomendacoes"], ensure_ascii=False),
            json.dumps(resultado["alertas"], ensure_ascii=False),
        )
    )
    conexao.commit()
    id_gerado = cursor.lastrowid
    conexao.close()
    return id_gerado


def listar_historico(caminho_db, limite=20):
    conexao = sqlite3.connect(caminho_db)
    conexao.row_factory = sqlite3.Row
    linhas = conexao.execute("SELECT * FROM analises ORDER BY id DESC LIMIT ?", (limite,)).fetchall()
    conexao.close()
    return [_linha_para_dict(linha) for linha in linhas]


def obter_analise_por_id(caminho_db, id_analise):
    conexao = sqlite3.connect(caminho_db)
    conexao.row_factory = sqlite3.Row
    linha = conexao.execute("SELECT * FROM analises WHERE id = ?", (id_analise,)).fetchone()
    conexao.close()
    return _linha_para_dict(linha) if linha else None


def _linha_para_dict(linha):
    chaves = linha.keys()
    return {
        "id": linha["id"], "data_hora": linha["data_hora"],
        "usuario_id": linha["usuario_id"] if "usuario_id" in chaves else "",
        "renda_mensal": linha["renda_mensal"], "nivel_endividamento": linha["nivel_endividamento"],
        "frequencia_poupanca": linha["frequencia_poupanca"],
        "transacoes": json.loads(linha["transacoes_json"]),
        "perfil_financeiro": linha["perfil_financeiro"], "probabilidade": linha["probabilidade"],
        "resumo_gastos": json.loads(linha["resumo_gastos_json"]),
        "recomendacoes": json.loads(linha["recomendacoes_json"]),
        "alertas": json.loads(linha["alertas_json"]) if "alertas_json" in chaves and linha["alertas_json"] else [],
    }


def processar_csv_em_lote(conteudo_csv, artefatos):
    df = pd.read_csv(io.BytesIO(conteudo_csv))
    colunas_faltando = [c for c in COLUNAS_CSV_OBRIGATORIAS if c not in df.columns]
    if colunas_faltando:
        raise ValueError(f"Colunas obrigatorias ausentes no CSV: {colunas_faltando}")
    resultados = []
    for usuario_id, grupo in df.groupby("usuario_id"):
        primeira_linha = grupo.iloc[0]
        dados_entrada = {
            "renda_mensal": float(primeira_linha["renda_mensal"]),
            "nivel_endividamento": float(primeira_linha["nivel_endividamento"]),
            "frequencia_poupanca": str(primeira_linha["frequencia_poupanca"]),
            "transacoes": [{"descricao": str(l["descricao"]), "valor": float(l["valor"])} for _, l in grupo.iterrows()],
        }
        resultado = analisar_financas(dados_entrada, artefatos)
        id_historico = salvar_analise_no_historico(CAMINHO_BANCO_DADOS, dados_entrada, resultado, usuario_id=str(usuario_id))
        resultados.append({"usuario_id": str(usuario_id), "id_historico": id_historico, **resultado})
    return resultados


artefatos_globais = {}


@asynccontextmanager
async def lifespan(app: FastAPI):
    print("Carregando modelos do OCI Object Storage...")
    artefatos_globais.update(baixar_e_carregar_artefatos())
    inicializar_banco_de_dados(CAMINHO_BANCO_DADOS)
    print("Modelos carregados e banco de historico pronto")
    yield
    artefatos_globais.clear()


app = FastAPI(title="API de Analise Financeira - G9 Team 20", lifespan=lifespan)


@app.get("/")
def raiz():
    return {"status": "API no ar", "modelos_carregados": len(artefatos_globais) > 0}


@app.post("/analise-financeira")
def analise_financeira(dados: AnaliseFinanceiraRequest):
    dados_dict = dados.model_dump()
    resultado = analisar_financas(dados_dict, artefatos_globais)
    id_historico = salvar_analise_no_historico(CAMINHO_BANCO_DADOS, dados_dict, resultado)
    return {**resultado, "id_historico": id_historico}


@app.post("/classificar-transacoes")
def classificar_transacoes(dados: ClassificarTransacoesRequest):
    transacoes_dict = [t.model_dump() for t in dados.transacoes]
    transacoes_classificadas = classificar_categorias_transacoes(transacoes_dict, artefatos_globais)
    return {"transacoes_classificadas": [
        {"descricao": t["descricao"], "valor": t["valor"], "categoria": t["categoria"]}
        for t in transacoes_classificadas
    ]}


@app.get("/explicabilidade/perfil")
def explicabilidade_perfil():
    return {"modelo": "Random Forest - Perfil Financeiro", "importancia_variaveis": obter_importancia_variaveis_perfil(artefatos_globais)}


@app.get("/explicabilidade/categoria")
def explicabilidade_categoria(top_n: int = Query(default=8, ge=1, le=20)):
    return {"modelo": "Regressao Logistica (TF-IDF) - Categoria de Transacao", "palavras_por_categoria": obter_palavras_influentes_categoria(artefatos_globais, top_n)}


@app.get("/historico")
def historico(limite: int = Query(default=20, ge=1, le=100)):
    return {"analises": listar_historico(CAMINHO_BANCO_DADOS, limite)}


@app.get("/historico/{id_analise}")
def historico_por_id(id_analise: int):
    analise = obter_analise_por_id(CAMINHO_BANCO_DADOS, id_analise)
    if analise is None:
        raise HTTPException(status_code=404, detail=f"Analise com id {id_analise} nao encontrada")
    return analise


@app.post("/analise-financeira/lote")
async def analise_financeira_lote(arquivo: UploadFile = File(...)):
    if not arquivo.filename.endswith(".csv"):
        raise HTTPException(status_code=400, detail="O arquivo deve ser um CSV (.csv)")
    conteudo = await arquivo.read()
    try:
        resultados = processar_csv_em_lote(conteudo, artefatos_globais)
    except ValueError as erro:
        raise HTTPException(status_code=422, detail=str(erro))
    except Exception as erro:
        raise HTTPException(status_code=422, detail=f"Erro ao processar o CSV: {str(erro)}")
    return {"total_processado": len(resultados), "resultados": resultados}


@app.get("/dashboard", response_class=HTMLResponse)
def dashboard():
    with open(CAMINHO_DASHBOARD_HTML, "r", encoding="utf-8") as f:
        return f.read()
'''


CONTEUDO_TESTES = '''
import sys
sys.path.insert(0, "/content")
import pytest
from fastapi.testclient import TestClient
from main import app


@pytest.fixture(scope="module")
def cliente():
    with TestClient(app) as c:
        yield c


def test_status_no_ar(cliente):
    assert cliente.get("/").status_code == 200


def test_dashboard_retorna_html(cliente):
    resposta = cliente.get("/dashboard")
    assert resposta.status_code == 200
    assert "Diagnostico de saude financeira" in resposta.text


def test_dashboard_contem_wizard_e_chart(cliente):
    resposta = cliente.get("/dashboard")
    assert "step-question" in resposta.text
    assert "chart.js" in resposta.text.lower()
    assert "fetch(" in resposta.text


def test_analise_ainda_funciona(cliente):
    dados = {"renda_mensal": 4500, "nivel_endividamento": 25, "frequencia_poupanca": "Media",
        "transacoes": [{"descricao": "Supermercado", "valor": 420}]}
    resposta = cliente.post("/analise-financeira", json=dados)
    assert resposta.status_code == 200


def test_historico_funciona(cliente):
    assert cliente.get("/historico").status_code == 200


def test_explicabilidade_funciona(cliente):
    assert cliente.get("/explicabilidade/perfil").status_code == 200
'''


def escrever_arquivos_locais() -> None:
    """Grava main.py, test_main.py e dashboard.html no ambiente do Colab."""
    with open("/content/main.py", "w", encoding="utf-8") as f:
        f.write(CONTEUDO_MAIN)
    with open("/content/test_main.py", "w", encoding="utf-8") as f:
        f.write(CONTEUDO_TESTES)
    with open("/content/dashboard.html", "wb") as f:
        f.write(base64.b64decode(DASHBOARD_HTML_B64))
    print("✅ main.py, test_main.py e dashboard.html escritos em /content")


def executar_suite_de_testes() -> bool:
    resultado = subprocess.run(
        [sys.executable, "-m", "pytest", "test_main.py", "-v"],
        capture_output=True, text=True, cwd="/content"
    )
    print(resultado.stdout)
    if resultado.stderr:
        print("⚠️", resultado.stderr)
    return resultado.returncode == 0


def verificar_ou_solicitar_chave(caminho_chave: str) -> bool:
    if os.path.exists(caminho_chave):
        print(f"✅ Chave '{caminho_chave}' já disponível nesta sessão")
        return True
    print(f"\\n🔑 Chave '{caminho_chave}' não encontrada — selecione o arquivo abaixo:")
    from google.colab import files
    files.upload()
    if os.path.exists(caminho_chave):
        print(f"✅ Chave '{caminho_chave}' recebida com sucesso")
        return True
    print(f"⚠️ Arquivo esperado '{caminho_chave}' não apareceu — confira o nome selecionado")
    return False


def enviar_e_reiniciar(ip_servidor, usuario, caminho_chave) -> None:
    """Envia main.py E dashboard.html (os dois arquivos) e reinicia a API."""
    chave = paramiko.RSAKey.from_private_key_file(caminho_chave)
    cliente_ssh = paramiko.SSHClient()
    cliente_ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cliente_ssh.connect(hostname=ip_servidor, username=usuario, pkey=chave, timeout=15)

    sftp = cliente_ssh.open_sftp()
    sftp.put("/content/main.py", "/home/ubuntu/main.py")
    sftp.put("/content/dashboard.html", "/home/ubuntu/dashboard.html")
    sftp.close()
    print("✅ main.py e dashboard.html enviados para o servidor")

    cliente_ssh.exec_command("pkill -f uvicorn")
    time.sleep(2)
    comando_subir = (
        "cd /home/ubuntu && source venv/bin/activate && "
        "nohup uvicorn main:app --host 0.0.0.0 --port 8000 > api.log 2>&1 < /dev/null &"
    )
    cliente_ssh.exec_command(comando_subir)
    time.sleep(3)
    cliente_ssh.close()
    print("✅ API reiniciada no servidor")


def confirmar_producao(ip_servidor) -> None:
    time.sleep(6)
    try:
        resposta = requests.get(f"http://{ip_servidor}:8000/dashboard", timeout=10)
        contem_wizard = "Diagnostico de saude financeira" in resposta.text
        contem_chart = "chart.js" in resposta.text.lower()
        print("🔍 Status:", resposta.status_code)
        print("🔍 Assistente presente?", contem_wizard)
        print("🔍 Chart.js carregado?", contem_chart)
        if resposta.status_code == 200 and contem_wizard and contem_chart:
            print("\\n🎉 Dashboard interativo CONFIRMADO em produção!")
            print(f"👉 Acesse: http://{ip_servidor}:8000/dashboard")
        else:
            print("\\n⚠️ Algo não bateu — revisar")
    except Exception as e:
        print(f"❌ Erro: {e}")


# ------------------------------------------------------------
# EXECUÇÃO COMPLETA
# ------------------------------------------------------------
IP_SERVIDOR = "140.238.178.157"
USUARIO = "ubuntu"
CAMINHO_CHAVE = "ssh-key-2026-08-01.key"

escrever_arquivos_locais()
testes_ok = executar_suite_de_testes()

if testes_ok:
    chave_disponivel = verificar_ou_solicitar_chave(CAMINHO_CHAVE)
    if chave_disponivel:
        print("\\n" + "=" * 60)
        print("🚀 Iniciando deploy para produção")
        print("=" * 60)
        enviar_e_reiniciar(IP_SERVIDOR, USUARIO, CAMINHO_CHAVE)
        confirmar_producao(IP_SERVIDOR)
    else:
        print("\\n❌ Deploy não realizado — chave SSH indisponível")
else:
    print("\\n❌ Testes locais falharam — deploy NÃO realizado, corrija antes de continuar")

✅ Dependências instaladas (incluindo paramiko)
✅ main.py, test_main.py e dashboard.html escritos em /content
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.14.2, langsmith-0.10.2, typeguard-4.5.2
collecting ... collected 6 items

test_main.py::test_status_no_ar PASSED                                   [ 16%]
test_main.py::test_dashboard_retorna_html PASSED                         [ 33%]
test_main.py::test_dashboard_contem_wizard_e_chart PASSED                [ 50%]
test_main.py::test_analise_ainda_funciona PASSED                         [ 66%]
test_main.py::test_historico_funciona PASSED                             [ 83%]
test_main.py::test_explicabilidade_funciona PASSED                       [100%]

============================== 6 passed in 5.89s ===============================

✅ Chave 'ssh-key-2026-08-01.key' 

In [26]:
import paramiko, time, requests

IP_SERVIDOR = "140.238.178.157"
USUARIO = "ubuntu"
CAMINHO_CHAVE = "ssh-key-2026-08-01.key"

chave = paramiko.RSAKey.from_private_key_file(CAMINHO_CHAVE)
cliente = paramiko.SSHClient()
cliente.set_missing_host_key_policy(paramiko.AutoAddPolicy())
cliente.connect(hostname=IP_SERVIDOR, username=USUARIO, pkey=chave, timeout=15)

print("=" * 60)
print("DIAGNÓSTICO")
print("=" * 60)

# A API esta rodando por dentro?
stdin, stdout, stderr = cliente.exec_command("curl -s -m 5 http://localhost:8000/ || echo 'SEM RESPOSTA LOCAL'")
print("1. API responde internamente?", stdout.read().decode().strip())

# Processo ativo?
stdin, stdout, stderr = cliente.exec_command("ps aux | grep '[u]vicorn' | head -2")
print("\n2. Processo uvicorn:", stdout.read().decode().strip() or "NENHUM")

# Regra de firewall da porta 8000 existe?
stdin, stdout, stderr = cliente.exec_command("sudo iptables -L INPUT -n --line-numbers | head -10")
print("\n3. Regras de firewall (INPUT):")
print(stdout.read().decode())

print("=" * 60)
print("APLICANDO CORREÇÃO")
print("=" * 60)

# Recria a regra da porta 8000 no topo da cadeia INPUT
cliente.exec_command("sudo iptables -I INPUT 1 -p tcp --dport 8000 -j ACCEPT")
time.sleep(2)

# Torna permanente (salva as regras atuais)
stdin, stdout, stderr = cliente.exec_command("sudo netfilter-persistent save 2>&1 || sudo iptables-save | sudo tee /etc/iptables/rules.v4 > /dev/null && echo 'Regras salvas'")
print("Persistência:", stdout.read().decode().strip())

# Confirma
stdin, stdout, stderr = cliente.exec_command("sudo iptables -L INPUT -n | grep 8000")
print("Regra 8000 agora:", stdout.read().decode().strip() or "AINDA AUSENTE")

cliente.close()

time.sleep(3)
print("\n" + "=" * 60)
try:
    r = requests.get(f"http://{IP_SERVIDOR}:8000/", timeout=10)
    print("🎉 API DE VOLTA AO AR:", r.json())
    print(f"👉 http://{IP_SERVIDOR}:8000/dashboard")
except Exception as e:
    print(f"❌ Ainda bloqueado: {e}")
    print("\n💡 Se persistir, verifique também a Security List da OCI no console —")
    print("   a regra de entrada da porta 8000 pode ter sido removida.")

DIAGNÓSTICO
1. API responde internamente? {"status":"API no ar","modelos_carregados":true}

2. Processo uvicorn: ubuntu      3066  3.4 15.8 528808 154548 ?       Ssl  04:49   0:06 /home/ubuntu/venv/bin/python3 /home/ubuntu/venv/bin/uvicorn main:app --host 127.0.0.1 --port 8000

3. Regras de firewall (INPUT):
Chain INPUT (policy ACCEPT)
num  target     prot opt source               destination         
1    ACCEPT     6    --  0.0.0.0/0            0.0.0.0/0            tcp dpt:80
2    ACCEPT     6    --  0.0.0.0/0            0.0.0.0/0            tcp dpt:8000
3    ACCEPT     0    --  0.0.0.0/0            0.0.0.0/0            state RELATED,ESTABLISHED
4    ACCEPT     1    --  0.0.0.0/0            0.0.0.0/0           
5    ACCEPT     0    --  0.0.0.0/0            0.0.0.0/0           
6    ACCEPT     6    --  0.0.0.0/0            0.0.0.0/0            state NEW tcp dpt:22
7    REJECT     0    --  0.0.0.0/0            0.0.0.0/0            reject-with icmp-host-prohibited

APLICANDO CORREÇÃO
Pe

In [27]:
# ============================================================
# Infraestrutura de Produção — systemd + nginx
# Módulo: Serviço persistente + Proxy reverso
# ============================================================
"""
Configura a API como serviço systemd (sobe sozinha no boot, reinicia se
cair) e instala o nginx como proxy reverso na porta 80 — eliminando a
necessidade de informar :8000 na URL e preparando terreno para HTTPS.
"""

import paramiko
import time
import requests

IP_SERVIDOR = "140.238.178.157"
USUARIO = "ubuntu"
CAMINHO_CHAVE = "ssh-key-2026-08-01.key"


def conectar(ip, usuario, caminho_chave):
    chave = paramiko.RSAKey.from_private_key_file(caminho_chave)
    cliente = paramiko.SSHClient()
    cliente.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cliente.connect(hostname=ip, username=usuario, pkey=chave, timeout=20)
    return cliente


def rodar(cliente, comando, descricao="", mostrar_saida=True):
    """Executa um comando remoto e exibe o resultado de forma legível."""
    stdin, stdout, stderr = cliente.exec_command(comando, timeout=120)
    codigo = stdout.channel.recv_exit_status()
    saida = stdout.read().decode().strip()
    erro = stderr.read().decode().strip()
    if descricao:
        marcador = "✅" if codigo == 0 else "⚠️"
        print(f"{marcador} {descricao}")
    if mostrar_saida and saida:
        print(f"   {saida[:400]}")
    if erro and codigo != 0:
        print(f"   ⚠️ {erro[:300]}")
    return codigo == 0


# ------------------------------------------------------------
# 1. SYSTEMD — serviço persistente
# ------------------------------------------------------------
ARQUIVO_SERVICO = """[Unit]
Description=API de Analise Financeira - G9 Team 20
After=network.target

[Service]
Type=simple
User=ubuntu
WorkingDirectory=/home/ubuntu
Environment="PATH=/home/ubuntu/venv/bin"
ExecStart=/home/ubuntu/venv/bin/uvicorn main:app --host 127.0.0.1 --port 8000
Restart=always
RestartSec=5
StandardOutput=append:/home/ubuntu/api.log
StandardError=append:/home/ubuntu/api.log

[Install]
WantedBy=multi-user.target
"""


# ------------------------------------------------------------
# 2. NGINX — proxy reverso
# ------------------------------------------------------------
CONFIG_NGINX = """server {
    listen 80;
    server_name _;

    location / {
        proxy_pass http://127.0.0.1:8000;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
        proxy_read_timeout 90s;
    }
}
"""


print("=" * 60)
print("CONFIGURANDO INFRAESTRUTURA DE PRODUÇÃO")
print("=" * 60)

cliente = conectar(IP_SERVIDOR, USUARIO, CAMINHO_CHAVE)

# --- Encerra o processo antigo (nohup) ---
rodar(cliente, "pkill -f uvicorn || true", "Processo antigo encerrado", mostrar_saida=False)
time.sleep(2)

# --- Cria o serviço systemd ---
comando_servico = f"echo '{ARQUIVO_SERVICO}' | sudo tee /etc/systemd/system/finance-api.service > /dev/null"
rodar(cliente, comando_servico, "Arquivo de serviço systemd criado", mostrar_saida=False)

rodar(cliente, "sudo systemctl daemon-reload", "systemd recarregado", mostrar_saida=False)
rodar(cliente, "sudo systemctl enable finance-api", "Serviço habilitado no boot", mostrar_saida=False)
rodar(cliente, "sudo systemctl restart finance-api", "Serviço iniciado", mostrar_saida=False)
time.sleep(8)
rodar(cliente, "sudo systemctl is-active finance-api", "Status do serviço:")

# --- Instala e configura o nginx ---
print("\n" + "-" * 60)
rodar(cliente, "sudo apt-get update -qq && sudo apt-get install -y nginx -qq", "nginx instalado", mostrar_saida=False)

comando_config = f"echo '{CONFIG_NGINX}' | sudo tee /etc/nginx/sites-available/finance-api > /dev/null"
rodar(cliente, comando_config, "Configuração do nginx criada", mostrar_saida=False)

rodar(cliente, "sudo ln -sf /etc/nginx/sites-available/finance-api /etc/nginx/sites-enabled/finance-api", "Site habilitado", mostrar_saida=False)
rodar(cliente, "sudo rm -f /etc/nginx/sites-enabled/default", "Site padrão removido", mostrar_saida=False)
rodar(cliente, "sudo nginx -t", "Configuração do nginx validada:")
rodar(cliente, "sudo systemctl restart nginx && sudo systemctl enable nginx", "nginx reiniciado e habilitado no boot", mostrar_saida=False)

# --- Libera a porta 80 no firewall interno ---
print("\n" + "-" * 60)
rodar(cliente, "sudo iptables -I INPUT 1 -p tcp --dport 80 -j ACCEPT", "Porta 80 liberada no iptables", mostrar_saida=False)
rodar(cliente, "sudo netfilter-persistent save", "Regras de firewall persistidas", mostrar_saida=False)
rodar(cliente, "sudo iptables -L INPUT -n | grep -E 'dpt:80|dpt:8000'", "Regras ativas:")

# --- Teste interno ---
print("\n" + "-" * 60)
rodar(cliente, "curl -s -m 5 http://localhost/ || echo 'sem resposta na porta 80'", "Teste interno (porta 80):")

cliente.close()

# --- Teste externo ---
print("\n" + "=" * 60)
print("TESTE EXTERNO")
print("=" * 60)
time.sleep(3)

for url, descricao in [
    (f"http://{IP_SERVIDOR}/", "porta 80 (nginx)"),
    (f"http://{IP_SERVIDOR}:8000/", "porta 8000 (direto)"),
]:
    try:
        r = requests.get(url, timeout=10)
        print(f"✅ {descricao}: {r.status_code} — {r.json()}")
    except Exception as e:
        print(f"❌ {descricao}: {type(e).__name__}")

print("\n💡 Se a porta 80 falhar externamente, falta liberar na Security List da OCI:")
print("   Console OCI → Networking → VCN → Security List → Add Ingress Rule")
print("   Source: 0.0.0.0/0 | Protocol: TCP | Destination Port: 80")

CONFIGURANDO INFRAESTRUTURA DE PRODUÇÃO
⚠️ Processo antigo encerrado
✅ Arquivo de serviço systemd criado
✅ systemd recarregado
✅ Serviço habilitado no boot
✅ Serviço iniciado
✅ Status do serviço:
   active

------------------------------------------------------------
✅ nginx instalado
✅ Configuração do nginx criada
✅ Site habilitado
✅ Site padrão removido
✅ Configuração do nginx validada:
✅ nginx reiniciado e habilitado no boot

------------------------------------------------------------
✅ Porta 80 liberada no iptables
✅ Regras de firewall persistidas
✅ Regras ativas:
   ACCEPT     6    --  0.0.0.0/0            0.0.0.0/0            tcp dpt:80
ACCEPT     6    --  0.0.0.0/0            0.0.0.0/0            tcp dpt:8000
ACCEPT     6    --  0.0.0.0/0            0.0.0.0/0            tcp dpt:80
ACCEPT     6    --  0.0.0.0/0            0.0.0.0/0            tcp dpt:8000

------------------------------------------------------------
✅ Teste interno (porta 80):
   {"status":"API no ar","modelos_c